In [1]:
import matplotlib.pyplot as pyplot
import numpy as np 
import pandas as pd 

In [2]:
df = pd.read_csv("epl_final.csv")

In [3]:
df.head()

,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,...,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,...,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,...,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,...,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,...,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,...,8,6,6,4,21,20,1,3,0,0


In [4]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [5]:
df.HomeTeam.nunique()

46

In [6]:
len(df)

9380

In [7]:
data = {}

for col in df.columns:
    data[col] = {'dtype' : df[col].dtype, 'nunique' : df[col].nunique(), 'null' : df[col].isnull().sum()}

In [8]:
data

{'Season': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 25,
  'null': np.int64(0)},
 'MatchDate': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 2599,
  'null': np.int64(0)},
 'HomeTeam': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 46,
  'null': np.int64(0)},
 'AwayTeam': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 46,
  'null': np.int64(0)},
 'FullTimeHomeGoals': {'dtype': dtype('int64'),
  'nunique': 10,
  'null': np.int64(0)},
 'FullTimeAwayGoals': {'dtype': dtype('int64'),
  'nunique': 10,
  'null': np.int64(0)},
 'FullTimeResult': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 3,
  'null': np.int64(0)},
 'HalfTimeHomeGoals': {'dtype': dtype('int64'),
  'nunique': 6,
  'null': np.int64(0)},
 'HalfTimeAwayGoals': {'dtype': dtype('int64'),
  'nunique': 6,
  'null': np.int64(0)},
 'HalfTimeResult': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nuniqu

In [9]:
df = df.sort_values("MatchDate")

In [10]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [11]:
teams = set(df.HomeTeam.unique())
away_teams = set(df.AwayTeam.unique())

In [12]:
len(teams), len(away_teams)

(46, 46)

In [13]:
teams == away_teams

True

# **ELO Rating**

Since the dataset is already pretty neat and clean, we will just create an ELO score for the team to help us rate how it has performed over the years. 

We will give all teams an initial ELO rating of 1500, and then adjust it and optimize it based on how each team performs. The following formula and process will be used to determine the right elo scores for each team. 


## **Elo Rating System**

Each team has a rating $R$.


### **Expected Score (Before Match)**

$$
E_A = \frac{1}{1 + 10^{\frac{R_B - R_A}{400}}}
$$

Where:

- $E_A$ = expected probability of Team A winning  
- $R_A$ = rating of Team A  
- $R_B$ = rating of Team B  

### **Rating Update (After Match)**

$$
R_A' = R_A + K \cdot (S_A - E_A)
$$

Where:

- $R_A'$ = updated rating for Team A  
- $K$ = learning rate / sensitivity factor (typically 20–40)  
- $S_A$ = actual result for Team A  

### **Match Result Encoding**

$$
S_A =
\begin{cases}
1 & \text{if Team A wins} \\
0.5 & \text{if draw} \\
0 & \text{if Team A loses}
\end{cases}
$$


### **Key Idea**

- If a team performs **better than expected**, its rating increases  
- If it performs **worse than expected**, its rating decreases

In [14]:
elo = {team: 1500 for team in teams}

In [15]:
def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

In [16]:
def get_score(result, is_home):
    if result == "H":
        return 1 if is_home else 0
    elif result == "A":
        return 0 if is_home else 1
    else:
        return 0.5

In [17]:
K = 20

home_elo = []
away_elo = []

for i, row in df.iterrows():
    home = row["HomeTeam"]
    away = row["AwayTeam"]

    Ra = elo[home]
    Rb = elo[away]

    home_elo.append(Ra)
    away_elo.append(Rb)

    Eh = expected_score(Ra, Rb)
    Ea = 1 - Eh

    Sh = get_score(row["FullTimeResult"], is_home=True)
    Sa = get_score(row["FullTimeResult"], is_home=False)

    elo[home] = Ra + K * (Sh - Eh)
    elo[away] = Rb + K * (Sa - Ea)

In [18]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [19]:
df.FullTimeResult.sample(10)

4839    A
7544    H
8723    H
1960    H
4730    H
4179    A
8879    H
1656    H
1600    H
3109    H
Name: FullTimeResult, dtype: str

In [20]:
df["Elo"] = df["HomeTeam"].map(elo)

In [21]:
df['HomeElo'] = home_elo
df['AwayElo'] = away_elo
df['EloDiff'] = df["HomeElo"] - df["AwayElo"]

In [22]:
df.Elo

0       1461.636013
1       1674.496189
2       1427.957841
3       1293.552821
4       1461.853400
           ...     
9377    1527.621265
9378    1674.496189
9375    1601.115637
9376    1600.432487
9379    1597.300961
Name: Elo, Length: 9380, dtype: float64

In [23]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff'],
      dtype='str')

In [24]:
df.Season.unique()

<StringArray>
['2000/01', '2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07',
 '2007/08', '2008/09', '2009/10', '2010/11', '2011/12', '2012/13', '2013/14',
 '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21',
 '2021/22', '2022/23', '2023/24', '2024/25']
Length: 25, dtype: str

# **Time Encoding**
Since we are using a GBM and not Timeseries, we need to give the model an idea of how to map time into its predictions. 

### **Encoding Seasons**
Since the seasons in the football matches are sequential, one follows another, we can use a primitive labeling of each season by just creating an enumerated list of seasons and then add it it a season encoded column that will be fed into the ML model. We will also add a season year column to help the model know which years predictions its making. 

### **Encoding Time in the current Season, like how far along are we since the season started**
Instead of mapping time by using a timestamp, which would be different for each game, we are going to use some sort of progress map that shows us from the season start, how many months we are into the game, I think we can also use days, but month and year can be sufficient for this model. 

In [25]:
season_map = {s: i for i, s in enumerate(sorted(df["Season"].unique()))}

df["SeasonEncoded"] = df["Season"].map(season_map)

In [26]:
season_map

{'2000/01': 0,
 '2001/02': 1,
 '2002/03': 2,
 '2003/04': 3,
 '2004/05': 4,
 '2005/06': 5,
 '2006/07': 6,
 '2007/08': 7,
 '2008/09': 8,
 '2009/10': 9,
 '2010/11': 10,
 '2011/12': 11,
 '2012/13': 12,
 '2013/14': 13,
 '2014/15': 14,
 '2015/16': 15,
 '2016/17': 16,
 '2017/18': 17,
 '2018/19': 18,
 '2019/20': 19,
 '2020/21': 20,
 '2021/22': 21,
 '2022/23': 22,
 '2023/24': 23,
 '2024/25': 24}

In [27]:
base_year = 2000
df["SeasonYear"] = df["SeasonEncoded"] + base_year

In [28]:
df.SeasonEncoded

0        0
1        0
2        0
3        0
4        0
        ..
9377    24
9378    24
9375    24
9376    24
9379    24
Name: SeasonEncoded, Length: 9380, dtype: int64

In [29]:
df.SeasonYear

0       2000
1       2000
2       2000
3       2000
4       2000
        ... 
9377    2024
9378    2024
9375    2024
9376    2024
9379    2024
Name: SeasonYear, Length: 9380, dtype: int64

In [30]:
df["MatchDate"] = pd.to_datetime(df["MatchDate"])
df = df.sort_values("MatchDate")

df["Month"] = df["MatchDate"].dt.month
df["Date"] = df["MatchDate"].dt.day

In [31]:
df.Month.unique()

array([ 8,  9, 10, 11, 12,  1,  2,  3,  4,  5,  6,  7], dtype=int32)

In [32]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff',
       'SeasonEncoded', 'SeasonYear', 'Month', 'Date'],
      dtype='str')

In [33]:
df.dtypes

Season                          str
MatchDate            datetime64[us]
HomeTeam                        str
AwayTeam                        str
FullTimeHomeGoals             int64
FullTimeAwayGoals             int64
FullTimeResult                  str
HalfTimeHomeGoals             int64
HalfTimeAwayGoals             int64
HalfTimeResult                  str
HomeShots                     int64
AwayShots                     int64
HomeShotsOnTarget             int64
AwayShotsOnTarget             int64
HomeCorners                   int64
AwayCorners                   int64
HomeFouls                     int64
AwayFouls                     int64
HomeYellowCards               int64
AwayYellowCards               int64
HomeRedCards                  int64
AwayRedCards                  int64
Elo                         float64
HomeElo                     float64
AwayElo                     float64
EloDiff                     float64
SeasonEncoded                 int64
SeasonYear                  

In [34]:
df.FullTimeResult.unique()

<StringArray>
['H', 'A', 'D']
Length: 3, dtype: str

In [35]:
mapping = {"H": 0, "D": 1, "A": 2}
df["Target"] = df["FullTimeResult"].map(mapping)

In [36]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff',
       'SeasonEncoded', 'SeasonYear', 'Month', 'Date', 'Target'],
      dtype='str')

## **Adding Points to the Dataset**

As is, we don't have a column for helping us know the points for each team per season. Since its likely that a winning team keeps winning subsequent matches, while a losing team is likely to keep losing for the rest of the team, adding this factor into the model would improve its likelihood of accuracy. Another aspect that we will add is simply evaluating how the team has been performing for the previous 5 games to evaluate how likely they keep staying on that trajectory. We are just feature engineering as many parameters as possible to evaluate how well the model performs out the gate. 

In [37]:
def get_home_points(result):
    if result == "H":
        return 3
    elif result == "D":
        return 1
    else:
        return 0

def get_away_points(result):
    if result == "A":
        return 3
    elif result == "D":
        return 1
    else:
        return 0

In [38]:
df["HomePoints"] = df["FullTimeResult"].map({"H": 3, "D": 1, "A": 0})
df["AwayPoints"] = df["FullTimeResult"].map({"A": 3, "D": 1, "H": 0})

In [39]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff',
       'SeasonEncoded', 'SeasonYear', 'Month', 'Date', 'Target', 'HomePoints',
       'AwayPoints'],
      dtype='str')

In [40]:
home = df[["MatchDate", "HomeTeam", "HomePoints"]].rename(
    columns={"HomeTeam": "Team", "HomePoints": "Points"}
)

away = df[["MatchDate", "AwayTeam", "AwayPoints"]].rename(
    columns={"AwayTeam": "Team", "AwayPoints": "Points"}
)

team_df = pd.concat([home, away])
team_df = team_df.sort_values("MatchDate")

In [42]:
team_df.head()

,MatchDate,Team,Points
0,2000-08-19,Charlton,3
3,2000-08-19,Southampton,1
4,2000-08-19,Everton,0
5,2000-08-19,Aston Villa,1
6,2000-08-19,Bradford,0


In [48]:
home

,MatchDate,Team,Points
0,2000-08-19,Charlton,3
1,2000-08-19,Chelsea,3
2,2000-08-19,Coventry,0
3,2000-08-19,Derby,1
4,2000-08-19,Leeds,3
...,...,...,...
9375,2025-05-04,Brentford,3
9376,2025-05-04,Brighton,1
9377,2025-05-04,West Ham,1
9378,2025-05-04,Chelsea,3


In [49]:
away

,MatchDate,Team,Points
0,2000-08-19,Man City,0
1,2000-08-19,West Ham,0
2,2000-08-19,Middlesbrough,3
3,2000-08-19,Southampton,1
4,2000-08-19,Everton,0
...,...,...,...
9375,2025-05-04,Man United,0
9376,2025-05-04,Newcastle,1
9377,2025-05-04,Tottenham,1
9378,2025-05-04,Liverpool,0


In [50]:
team_df = pd.concat([home, away])

In [51]:
team_df = team_df.sort_values("MatchDate").reset_index(drop=True)

In [52]:
team_df["FormPoints_5"] = (
    team_df.groupby("Team")["Points"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)

In [53]:
team_df

,MatchDate,Team,Points,FormPoints_5
0,2000-08-19,Charlton,3,3.0
1,2000-08-19,Southampton,1,1.0
2,2000-08-19,Everton,0,0.0
3,2000-08-19,Aston Villa,1,1.0
4,2000-08-19,Bradford,0,0.0
...,...,...,...,...
18755,2025-05-04,Brentford,3,2.2
18756,2025-05-04,Liverpool,0,1.8
18757,2025-05-04,Tottenham,1,0.8
18758,2025-05-05,Crystal Palace,1,0.6


In [59]:
home_form = team_df.rename(columns={
    "Team": "HomeTeam",
    "FormPoints_5": "HomeFormPoints_5"
})[["MatchDate", "HomeTeam", "HomeFormPoints_5"]]

In [60]:
team_df

,MatchDate,Team,Points,FormPoints_5
0,2000-08-19,Charlton,3,3.0
1,2000-08-19,Southampton,1,1.0
2,2000-08-19,Everton,0,0.0
3,2000-08-19,Aston Villa,1,1.0
4,2000-08-19,Bradford,0,0.0
...,...,...,...,...
18755,2025-05-04,Brentford,3,2.2
18756,2025-05-04,Liverpool,0,1.8
18757,2025-05-04,Tottenham,1,0.8
18758,2025-05-05,Crystal Palace,1,0.6


In [62]:
away_form = team_df.rename(columns={
    "Team": "AwayTeam",
    "FormPoints_5": "AwayFormPoints_5"
})[["MatchDate", "AwayTeam", "AwayFormPoints_5"]]

In [56]:
team_df.columns

Index(['MatchDate', 'Team', 'Points', 'FormPoints_5'], dtype='str')

In [63]:
df = df.merge(home_form, on=["MatchDate", "HomeTeam"], how="left")
df = df.merge(away_form, on=["MatchDate", "AwayTeam"], how="left")

In [65]:
df

,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,...,EloDiff,SeasonEncoded,SeasonYear,Month,Date,Target,HomePoints,AwayPoints,HomeFormPoints_5,AwayFormPoints_5
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,...,0.000000,0,2000,8,19,0,3,0,3.0,0.0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,...,0.000000,0,2000,8,19,0,3,0,3.0,0.0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,...,0.000000,0,2000,8,19,2,0,3,0.0,3.0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,...,0.000000,0,2000,8,19,1,1,1,1.0,1.0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,...,0.000000,0,2000,8,19,0,3,0,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9375,2024/25,2025-05-04,Brentford,Man United,4,3,H,2,1,H,...,16.559438,24,2024,5,4,0,3,0,2.2,0.4
9376,2024/25,2025-05-04,Brighton,Newcastle,1,1,D,1,0,H,...,-72.600945,24,2024,5,4,1,1,1,1.0,2.0
9377,2024/25,2025-05-04,West Ham,Tottenham,1,1,D,1,1,D,...,-17.375446,24,2024,5,4,1,1,1,0.6,0.8
9378,2024/25,2025-05-04,Chelsea,Liverpool,3,1,H,1,0,H,...,-150.804752,24,2024,5,4,0,3,0,2.2,1.8


In [66]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff',
       'SeasonEncoded', 'SeasonYear', 'Month', 'Date', 'Target', 'HomePoints',
       'AwayPoints', 'HomeFormPoints_5', 'AwayFormPoints_5'],
      dtype='str')

In [68]:
feature_cols = ['EloDiff', 'HomeElo', 'AwayElo', 'HomeFormPoints_5', 'AwayFormPoints_5', 'SeasonEncoded', 'SeasonYear', 'Month', 'Date']

In [69]:
features = df[feature_cols]

In [70]:
features

,EloDiff,HomeElo,AwayElo,HomeFormPoints_5,AwayFormPoints_5,SeasonEncoded,SeasonYear,Month,Date
0,0.000000,1500.000000,1500.000000,3.0,0.0,0,2000,8,19
1,0.000000,1500.000000,1500.000000,3.0,0.0,0,2000,8,19
2,0.000000,1500.000000,1500.000000,0.0,3.0,0,2000,8,19
3,0.000000,1500.000000,1500.000000,1.0,1.0,0,2000,8,19
4,0.000000,1500.000000,1500.000000,3.0,0.0,0,2000,8,19
...,...,...,...,...,...,...,...,...,...
9375,16.559438,1591.591895,1575.032457,2.2,0.4,24,2024,5,4
9376,-72.600945,1598.372756,1670.973701,1.0,2.0,24,2024,5,4
9377,-17.375446,1527.121576,1544.497021,0.6,0.8,24,2024,5,4
9378,-150.804752,1660.409177,1811.213929,2.2,1.8,24,2024,5,4


In [71]:
features.to_csv('features.csv')